# 00 — Kiểm tra dữ liệu ProPara (data audit) · `propara_audit_v1`

**Vị trí trong pipeline:** Giai đoạn 1 (D1) của kế hoạch Go/No-Go. Là notebook đầu tiên; **01 (bộ chấm + heuristic) và mọi notebook Qwen đều phụ thuộc vào output của notebook này.**

**Mục tiêu:** tải ProPara gốc (bản EMNLP'18 `grids.v1` mà báo cáo dùng), chuẩn hoá, suy nhãn CREATE/DESTROY/MOVE/NONE, cờ nhắc, thay đổi ngầm, 533 cặp vàng, điểm quyết định ngầm. Sau đó **đối chiếu từng con số trong báo cáo 16/10/2026**, đối chiếu với đáp án của bộ chấm chính thức (`aristo-leaderboard`) và xuất dữ liệu chuẩn có checksum.

| Mục | Giá trị |
|---|---|
| Accelerator | **None (CPU)**. Không cần GPU, nên **tắt GPU** để giữ quota |
| Internet | **On** (để tải dữ liệu từ GitHub, ghim theo commit). Nếu không bật được thì dùng phương án Dataset (xem dưới) |
| Input | Không bắt buộc. Phương án dự phòng: Dataset riêng chứa `propara-raw-v1.zip` (đã giải nén) |
| Thư viện | Chỉ thư viện chuẩn của Python + `matplotlib` (Kaggle có sẵn). Không cần `pip install` |
| Thời gian | Toàn bộ dữ liệu chỉ ~1 MB; trên máy CPU thử nghiệm, notebook chạy xong dưới 1 phút (chưa đo trên Kaggle) |
| Output | `/kaggle/working/propara_audit_v1/` và `/kaggle/working/propara_audit_v1.zip` |

**Cách chạy:** `Run All`. Đọc bảng **CỔNG NGHIỆM THU** ở ô cuối. Nếu ghi `GATE_STATUS = PASS` thì `Save Version` → `Save & Run All (Commit)`.

**Điều kiện thành công:** không có dòng `FAIL`. Các dòng `WARN` là phát hiện cần ghi nhận, **không chặn** bước tiếp theo.

**Gửi lại cho trợ lý:** toàn bộ output của **ô cuối cùng** (khối `=== GỬI LẠI ===`) và file `propara_audit_v1/reports/audit_summary.json`.

> Quy ước trong notebook: `sentence_id` đánh số từ 1; `states[i][0]` là trạng thái ban đầu, `states[i][t]` là trạng thái **sau** câu *t*. Thực thể được định danh bằng **chỉ số** (không bằng tên), vì ProPara có thực thể trùng tên.

## 1. Cấu hình

In [1]:
import os, sys, json, time, platform, shutil, hashlib, csv, zipfile, random, statistics, urllib.request
from collections import Counter, defaultdict
from pathlib import Path

SMOKE_TEST     = False   # True: chỉ chạy 15 đoạn/tập để thử luồng (bỏ qua đối chiếu số liệu báo cáo)
ALLOW_DOWNLOAD = True    # cho phép tải từ GitHub nếu không thấy dữ liệu trong /kaggle/input
SEED           = 2026
RUN_TAG        = "propara_audit_v1"

# Biến môi trường chỉ dùng khi chạy NGOÀI Kaggle (kiểm thử cục bộ); trên Kaggle để mặc định.
INPUT_ROOT = Path(os.environ.get("PROPARA_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT  = Path(os.environ.get("PROPARA_WORK_ROOT", "/kaggle/working"))
OUT = WORK_ROOT / RUN_TAG

PIN_PROPARA = "51a3cccbedcbe5f92276f9925e0ce1e6ec4247bb"   # allenai/propara (2019-08-30)
PIN_LEADER  = "60d5be31b5a7d36f29d13223ece29a8d2bfa8b5f"   # allenai/aristo-leaderboard (2021-05-17)
_GH = "https://raw.githubusercontent.com/allenai"

# tên file cục bộ -> (URL ghim commit, sha256, các hậu tố đường dẫn chấp nhận khi dò /kaggle/input)
RAW_FILES = {
    "grids.v1.train.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.train.json",
        "6dc048d09c532ef906ba37d5ab814f54c54e56f230184029912eabe415de225d", ["grids.v1.train.json"]),
    "grids.v1.dev.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.dev.json",
        "580ddc66d86a89c747d3e7f2da5cca0eee95444cca66b60b345fe29c93c3fef6", ["grids.v1.dev.json"]),
    "grids.v1.test.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.test.json",
        "cf239a45d3eb203aba09ec392cffc81812e5c6fde69d782b01d068efd9e6f370", ["grids.v1.test.json"]),
    "naacl18.partition.paraid.tsv": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/naacl18.partition.paraid.tsv",
        "9fecf07296c58b195ae778943fc0b4d38ea489ec71e19045bd8730460175de44", ["naacl18.partition.paraid.tsv"]),
    "gold-full-grids.v3.tsv": (f"{_GH}/propara/{PIN_PROPARA}/data/naacl18/gold-full-grids.v3.tsv",
        "8eff6e8fb066869d9d0c0fe18e3d994eb3078caa0a3a1be98f716e8a79dbe0aa", ["gold-full-grids.v3.tsv"]),
    "leaderboard_answers_train.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/train/answers.tsv",
        "2f2b014aa071583f9180cd8487c855438821abe1c9aae5474ac685854934e8cd", ["leaderboard_answers_train.tsv", "train/answers.tsv"]),
    "leaderboard_answers_dev.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/dev/answers.tsv",
        "a496afe5bb2b81ad79b0da7a935f81d5d0a6b4cefc89d865ba09fece38a30217", ["leaderboard_answers_dev.tsv", "dev/answers.tsv"]),
    "leaderboard_answers_test.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/test/answers.tsv",
        "cf1a71f38ccad7609e54b1dcf5cbc6e342ea3d2513490c0c09883d5bd80625d6", ["leaderboard_answers_test.tsv", "test/answers.tsv"]),
}
SPLITS = ("train", "dev", "test")

# Con số trong báo cáo 16/10/2026 (trang ghi kèm), dùng để đối chiếu.
REPORT = {
    "n_paragraphs": 488, "n_sentences": 3302, "n_entities": 1915,                       # tr.1
    "table1": {  # (ngầm, tổng) theo tập và nhãn — tr.2, Bảng 1
        "train": {"CREATE": (78, 747), "DESTROY": (81, 502), "MOVE": (355, 1211)},
        "dev":   {"CREATE": (11, 85),  "DESTROY": (15, 68),  "MOVE": (41, 157)},
        "test":  {"CREATE": (15, 128), "DESTROY": (22, 121), "MOVE": (53, 152)},
    },
    "gold_pairs": 533, "gold_pairs_split": {"train": 378, "dev": 58, "test": 97},       # tr.2, (3)
    "decision_points": {"DESTROY": 57, "CREATE": 76},                                     # tr.2
    "avg_candidates": {"DESTROY": 2.6, "CREATE": 2.9},                                    # tr.3
    "random_acc_pct": {"DESTROY": 51, "CREATE": 40},                                      # tr.3, Bảng 3
    "lift": {  # (lift, số ô) — tr.2, Bảng 2; hàng = nhãn ngầm A, cột = tín hiệu B
        ("CREATE", "CREATE"): (0.3, 10), ("CREATE", "DESTROY"): (6.6, 63), ("CREATE", "MOVE"): (0.3, 10),
        ("DESTROY", "CREATE"): (9.0, 85), ("DESTROY", "DESTROY"): (1.4, 19), ("DESTROY", "MOVE"): (0.0, 2),
        ("MOVE", "CREATE"): (1.0, 97), ("MOVE", "DESTROY"): (1.3, 69), ("MOVE", "MOVE"): (1.2, 141),
    },
    "pp1_label_questions": 52936, "pp1_pair_questions": 48070,                            # tr.4, (6)(7)
    "pp2_kept": 11144, "pp2_gold_kept": 533,                                              # tr.4, (12)
    "cooc_gold_pct": 74.7, "cooc_src_implicit_pct": 18.9,                                 # tr.4
    "mean_entities": 3.9,                                                                 # tr.6
}
# Số đoạn theo tập lấy từ file phân chia chính thức (báo cáo không nêu)
EXPECTED_SPLIT_PARAS = {"train": 391, "dev": 43, "test": 54}

random.seed(SEED)
OUT.mkdir(parents=True, exist_ok=True)
for sub in ["raw", "data/processed", "data/splits", "data/audit", "reports/figures"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
T0 = time.time()
print("OUT =", OUT, "| SMOKE_TEST =", SMOKE_TEST)

OUT = C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\runs\r1792352508\propara_audit_v1 | SMOKE_TEST = False


## 2. Kiểm tra môi trường

In [2]:
def _internet_ok(url="https://raw.githubusercontent.com", timeout=6):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception as e:  # mạng tắt hoặc bị chặn
        return f"không truy cập được: {type(e).__name__}"

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "on_kaggle": Path("/kaggle").exists() and "PROPARA_INPUT_ROOT" not in os.environ,
    "cpu_count": os.cpu_count(),
    "disk_free_gb": round(shutil.disk_usage(WORK_ROOT).free / 1e9, 1),
    "gpu_present": bool(shutil.which("nvidia-smi")),
    "internet": _internet_ok(),
}
try:
    with open("/proc/meminfo") as f:
        ENV["ram_total_gb"] = round(int(f.readline().split()[1]) / 1e6, 1)
except OSError:
    ENV["ram_total_gb"] = None
try:
    import matplotlib
    ENV["matplotlib"] = matplotlib.__version__
except ImportError:
    ENV["matplotlib"] = None
for k, v in ENV.items():
    print(f"{k:14s}: {v}")
if ENV["gpu_present"]:
    print("\n[CẢNH BÁO] Đang bật GPU. Notebook này chỉ cần CPU, nên tắt GPU (Settings → Accelerator → None) để giữ quota.")
if ENV["disk_free_gb"] < 1:
    raise RuntimeError("Dung lượng trống < 1 GB trong thư mục làm việc.")

python        : 3.14.4
platform      : Windows-11-10.0.26300-SP0
on_kaggle     : False
cpu_count     : 12
disk_free_gb  : 291.6
gpu_present   : True
internet      : True
ram_total_gb  : None
matplotlib    : 3.10.9

[CẢNH BÁO] Đang bật GPU. Notebook này chỉ cần CPU, nên tắt GPU (Settings → Accelerator → None) để giữ quota.


## 3. Mô-đun lõi `propara_core` (nhúng nguyên văn, ghi ra output để notebook sau dùng lại)

In [3]:
CORE_SRC = r'''
"""propara_core — chuẩn hoá ProPara, suy nhãn, cờ nhắc, cặp vàng, điểm quyết định.

Phiên bản: core_v1 (notebook 00_propara_data_audit). Chỉ dùng thư viện chuẩn của Python.

Quy ước chỉ số:
  * states[i] có độ dài T+1: states[i][0] là trạng thái ban đầu, states[i][t] là trạng thái SAU câu t.
  * labels[i], mentions[i], implicit[i] có độ dài T; phần tử [t-1] ứng với câu t (đánh số từ 1).
  * sentence_id luôn đánh số từ 1 như bộ chấm chính thức của ProPara.
"""
from __future__ import annotations

import hashlib
import json
import re
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional, Tuple

CORE_VERSION = "core_v1"

NONE, CREATE, DESTROY, MOVE = "NONE", "CREATE", "DESTROY", "MOVE"
LABELS = (CREATE, DESTROY, MOVE, NONE)
NO_LOC = "-"   # không tồn tại
UNK_LOC = "?"  # tồn tại nhưng không rõ vị trí

# Từ rỗng bỏ qua khi xét cờ nhắc. Danh sách này được KHOÁ cùng core_v1;
# với danh sách này + MENTION_MODE="report" ta tái hiện đúng Bảng 1 của báo cáo.
STOPWORDS = frozenset(
    "a an the of in on at to for from by with and or its it this that these those "
    "is are be into onto some any other another their his her your our".split()
)

_TOKEN_RE = re.compile(r"[a-z0-9]+")


# ----------------------------------------------------------------------------- tiện ích
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def tokenize(text: str) -> List[str]:
    return _TOKEN_RE.findall(text.lower())


def split_aliases(participant: str) -> List[str]:
    """'water vapor ; vapor' -> ['water vapor', 'vapor'] (giống evaluator: tách theo ';')."""
    return [p.strip() for p in participant.split(";") if p.strip()]


# ----------------------------------------------------------------------------- nhãn
def derive_label(before: str, after: str) -> str:
    """Công thức (1) của báo cáo."""
    if before == after:
        return NONE
    if before == NO_LOC and after != NO_LOC:
        return CREATE
    if before != NO_LOC and after == NO_LOC:
        return DESTROY
    return MOVE


def exists(state: str) -> bool:
    """'?' = tồn tại nhưng không rõ vị trí; chỉ '-' là không tồn tại."""
    return state != NO_LOC


VALID_LABELS = {False: {CREATE, NONE}, True: {DESTROY, MOVE, NONE}}  # công thức (15)


def count_constraint_violations(labels_by_entity: List[List[str]],
                                init_exists: List[bool]) -> Tuple[int, int]:
    """Số ô vi phạm ràng buộc tồn tại khi trạng thái TỰ TIẾN HOÁ theo nhãn dự đoán (công thức 28).

    Trả về (số ô vi phạm, số ô). Khi vi phạm, trạng thái vẫn cập nhật theo nhãn
    (CREATE -> tồn tại, DESTROY -> không tồn tại) để không che lỗi tiếp theo.
    """
    bad = total = 0
    for labs, e in zip(labels_by_entity, init_exists):
        for y in labs:
            total += 1
            if y not in VALID_LABELS[e]:
                bad += 1
            if y == CREATE:
                e = True
            elif y == DESTROY:
                e = False
    return bad, total


# ----------------------------------------------------------------------------- cờ nhắc
def _variants(word: str, mode: str) -> set:
    v = {word, word + "s"}
    if word.endswith("s"):
        v.add(word[:-1])
    if mode == "extended":  # nhạy hơn với số nhiều -es/-ies (dùng cho phân tích độ nhạy)
        v.add(word + "es")
        if word.endswith("es"):
            v.add(word[:-2])
        if word.endswith("ies"):
            v.add(word[:-3] + "y")
        if word.endswith("y"):
            v.add(word[:-1] + "ies")
    return v


def is_mentioned(participant: str, sentence: str, mode: str = "report") -> bool:
    """m_it = 1 nếu có ít nhất một từ nội dung của tên thực thể (mọi tên gọi khác,
    dạng số ít/số nhiều) xuất hiện trong câu.

    mode="report": khớp w, w+s, w bỏ 's' — tái hiện đúng Bảng 1 của báo cáo.
    mode="extended": thêm -es/-ies (echo/echoes, virus/viruses) — chỉ để đo độ nhạy.
    """
    if mode not in ("report", "extended"):
        raise ValueError(f"mode không hợp lệ: {mode}")
    sent_tokens = set(tokenize(sentence))
    for alias in split_aliases(participant):
        for w in tokenize(alias):
            if w in STOPWORDS:
                continue
            if _variants(w, mode) & sent_tokens:
                return True
    return False


# ----------------------------------------------------------------------------- bản ghi chuẩn
def build_record(raw: dict, split: str, topic: Optional[str] = None,
                 mention_mode: str = "report") -> dict:
    """Chuyển một dòng grids.v1.*.json thành bản ghi chuẩn (không sửa nhãn gốc)."""
    sents = list(raw["sentence_texts"])
    parts = list(raw["participants"])
    states = [list(s) for s in raw["states"]]
    T = len(sents)
    labels, mentions, implicit, existence = [], [], [], []
    for i, p in enumerate(parts):
        st = states[i]
        labs = [derive_label(st[t - 1], st[t]) for t in range(1, len(st))]
        men = [is_mentioned(p, sents[t - 1], mention_mode) for t in range(1, T + 1)] if len(st) == T + 1 else []
        labels.append(labs)
        mentions.append(men)
        implicit.append([(y != NONE) and (not m) for y, m in zip(labs, men)])
        existence.append([exists(s) for s in st])
    return {
        "paragraph_id": str(raw["para_id"]),
        "split": split,
        "topic": topic,
        "sentences": sents,
        "entities": parts,
        "entity_aliases": [split_aliases(p) for p in parts],
        "states": states,
        "existence": existence,
        "labels": labels,
        "mentions": mentions,
        "implicit": implicit,
        "mention_mode": mention_mode,
        "core_version": CORE_VERSION,
    }


def validate_record(rec: dict) -> List[str]:
    """Kiểm tra cấu trúc; trả về danh sách lỗi (rỗng = hợp lệ)."""
    issues = []
    pid = rec["paragraph_id"]
    T, n = len(rec["sentences"]), len(rec["entities"])
    if T == 0:
        issues.append(f"{pid}: không có câu")
    if n == 0:
        issues.append(f"{pid}: không có thực thể")
    if len(rec["states"]) != n:
        issues.append(f"{pid}: số hàng states ({len(rec['states'])}) != số thực thể ({n})")
    for t, s in enumerate(rec["sentences"], 1):
        if not s or not s.strip():
            issues.append(f"{pid}: câu {t} rỗng")
    lowered = [p.strip().lower() for p in rec["entities"]]
    for p, c in Counter(lowered).items():
        if c > 1:
            issues.append(f"{pid}: thực thể trùng tên '{p}'")
    for i, st in enumerate(rec["states"]):
        if len(st) != T + 1:
            issues.append(f"{pid}: thực thể {i} có {len(st)} trạng thái, cần {T + 1}")
        for t, v in enumerate(st):
            if not isinstance(v, str) or v.strip() == "":
                issues.append(f"{pid}: trạng thái rỗng tại thực thể {i}, bước {t}")
            elif v != v.strip():
                issues.append(f"{pid}: trạng thái có khoảng trắng thừa '{v}' (thực thể {i}, bước {t})")
    return issues


# ----------------------------------------------------------------------------- cặp vàng
def destroyed_created_at(rec: dict, t: int) -> Tuple[List[int], List[int]]:
    D = [i for i in range(len(rec["entities"])) if rec["labels"][i][t - 1] == DESTROY]
    C = [i for i in range(len(rec["entities"])) if rec["labels"][i][t - 1] == CREATE]
    return D, C


def gold_pairs_report(rec: dict) -> List[dict]:
    """Công thức (3): GoldPairs(t) = D_t x C_t. Đây là nhãn SUY TỪ TRẠNG THÁI,
    không phải quan hệ nhân quả đã được người xác nhận (pair_gold_type)."""
    out = []
    for t in range(1, len(rec["sentences"]) + 1):
        D, C = destroyed_created_at(rec, t)
        for x in D:
            for y in C:
                out.append({
                    "paragraph_id": rec["paragraph_id"],
                    "split": rec["split"],
                    "sentence_id": t,
                    "source": rec["entities"][x],
                    "target": rec["entities"][y],
                    "source_idx": x,
                    "target_idx": y,
                    "source_location_before": rec["states"][x][t - 1],
                    "target_location_after": rec["states"][y][t],
                    "source_implicit": bool(rec["implicit"][x][t - 1]),
                    "target_implicit": bool(rec["implicit"][y][t - 1]),
                    "n_destroy_in_sentence": len(D),
                    "n_create_in_sentence": len(C),
                    "pair_gold_type": "derived_from_states",
                })
    return out


def official_q3_conversions(rec: dict) -> List[dict]:
    """Bản sao logic Process.conversions() của evaluator chính thức
    (allenai/aristo-leaderboard/propara, commit 60d5be3), GIỮ NGUYÊN cả điều kiện
    `step_id < num_steps - 1`. Mỗi câu cho tối đa MỘT conversion, gộp mọi thực thể
    bị huỷ/tạo bằng ' AND '; có luật ghép với câu kế tiếp (rule b).
    Notebook 01 sẽ đối chiếu hàm này với mã evaluator gốc."""
    T = len(rec["sentences"])
    names = [" OR ".join(split_aliases(p)) for p in rec["entities"]]

    def loc(v):  # evaluator đổi '?' -> 'unk', '-' -> 'null'
        return {"?": "unk", "-": "null"}.get(v, v)

    def created(t):
        idx = [i for i in range(len(names)) if rec["states"][i][t - 1] == NO_LOC and rec["states"][i][t] != NO_LOC]
        return [names[i] for i in idx], [loc(rec["states"][i][t]) for i in idx]

    def destroyed(t):
        idx = [i for i in range(len(names)) if rec["states"][i][t - 1] != NO_LOC and rec["states"][i][t] == NO_LOC]
        return [names[i] for i in idx], [loc(rec["states"][i][t - 1]) for i in idx]

    conv = []
    for t in range(1, T + 1):
        c, cl = created(t)
        d, dl = destroyed(t)
        if c and d:
            conv.append({"step_id": t, "destroyed": " AND ".join(d), "created": " AND ".join(c),
                         "locations": sorted(set(cl + dl)), "rule": "same_step"})
        elif d and t < T - 1:
            c2, cl2 = created(t + 1)
            d2, _ = destroyed(t + 1)
            cnd = [x for x in c2 if x not in set(d)]
            if not d2 and cnd:
                conv.append({"step_id": t, "destroyed": " AND ".join(d), "created": " AND ".join(sorted(set(cnd))),
                             "locations": sorted(set(cl2 + dl)), "rule": "next_step_create"})
        elif c and t < T - 1:
            c2, _ = created(t + 1)
            d2, dl2 = destroyed(t + 1)
            dnc = [x for x in d2 if x not in set(c)]
            if not c2 and dnc:
                conv.append({"step_id": t, "destroyed": " AND ".join(sorted(set(dnc))), "created": " AND ".join(c),
                             "locations": sorted(set(cl + dl2)), "rule": "next_step_destroy"})
    return conv


def to_action_rows(rec: dict) -> List[Tuple[str, int, str, str, str, str]]:
    """Dạng 'action file' của evaluator: (process_id, step, participant, action, before, after)."""
    rows = []
    for t in range(1, len(rec["sentences"]) + 1):
        for i, p in enumerate(rec["entities"]):
            rows.append((rec["paragraph_id"], t, p, rec["labels"][i][t - 1],
                         rec["states"][i][t - 1], rec["states"][i][t]))
    return rows


# ----------------------------------------------------------------------------- điểm quyết định ngầm
def decision_points(rec: dict) -> List[dict]:
    """Mục 2.2: bộ (đoạn, câu, nhãn) có >=1 ô vàng ngầm và >=2 ứng viên.
    Ứng viên DESTROY = thực thể đang tồn tại trước câu; CREATE = chưa tồn tại;
    thực thể vàng ĐƯỢC NHẮC bị loại khỏi tập ứng viên."""
    out = []
    n = len(rec["entities"])
    for t in range(1, len(rec["sentences"]) + 1):
        for lab in (DESTROY, CREATE):
            need_exist = lab == DESTROY
            gold = [i for i in range(n) if rec["labels"][i][t - 1] == lab]
            gold_implicit = [i for i in gold if rec["implicit"][i][t - 1]]
            gold_mentioned = [i for i in gold if not rec["implicit"][i][t - 1]]
            if not gold_implicit:
                continue
            cands = [i for i in range(n)
                     if rec["existence"][i][t - 1] == need_exist and i not in gold_mentioned]
            if len(cands) < 2:
                continue
            out.append({
                "paragraph_id": rec["paragraph_id"], "split": rec["split"], "topic": rec.get("topic"),
                "sentence_id": t, "label": lab,
                "candidates": cands, "gold": gold_implicit,
                "n_candidates": len(cands), "n_gold": len(gold_implicit),
                "random_acc": len(gold_implicit) / len(cands),
            })
    return out


# ----------------------------------------------------------------------------- PP2 / PP1 (đếm)
def pp2_counts(rec: dict) -> Dict[str, int]:
    """Công thức (7) và (12): mọi cặp có thứ tự (X != Y) mỗi câu, và số cặp còn lại
    sau lọc bằng TỒN TẠI VÀNG (X tồn tại trước câu, Y chưa tồn tại). Đây là MỨC TRẦN."""
    T, n = len(rec["sentences"]), len(rec["entities"])
    total = T * n * (n - 1)
    kept = 0
    gold = {(g["source_idx"], g["target_idx"], g["sentence_id"]) for g in gold_pairs_report(rec)}
    gold_kept = 0
    for t in range(1, T + 1):
        ex = [rec["existence"][i][t - 1] for i in range(n)]
        for x in range(n):
            for y in range(n):
                if x != y and ex[x] and not ex[y]:
                    kept += 1
                    if (x, y, t) in gold:
                        gold_kept += 1
    return {"all_pairs": total, "kept_gold_existence": kept,
            "gold_pairs": len(gold), "gold_kept": gold_kept, "label_questions": 4 * T * n}


def cooccurrence_filter_coverage(rec: dict) -> Dict[str, int]:
    """Thử tái hiện câu 'lọc theo đồng xuất hiện trong câu chỉ giữ 74,7% cặp vàng và 18,9%
    cặp có nguồn ngầm' (mục 4.2). Báo cáo KHÔNG định nghĩa rõ, nên đo hai cách hiểu:
      same_sentence: X và Y cùng được nhắc trong chính câu t;
      any_sentence:  X và Y cùng được nhắc trong ít nhất một câu nào đó của đoạn."""
    out = Counter()
    T = len(rec["sentences"])
    for g in gold_pairs_report(rec):
        x, y, t = g["source_idx"], g["target_idx"], g["sentence_id"]
        same = rec["mentions"][x][t - 1] and rec["mentions"][y][t - 1]
        anys = any(rec["mentions"][x][k] and rec["mentions"][y][k] for k in range(T))
        out["gold"] += 1
        out["same_sentence"] += int(same)
        out["any_sentence"] += int(anys)
        if g["source_implicit"]:
            out["gold_src_implicit"] += 1
            out["same_sentence_src_implicit"] += int(same)
            out["any_sentence_src_implicit"] += int(anys)
    return dict(out)


# ----------------------------------------------------------------------------- lift (Bảng 2)
def lift_table(records: Iterable[dict]) -> Dict[Tuple[str, str], Dict[str, float]]:
    """Công thức (2). Ô hợp lệ: CREATE -> thực thể chưa tồn tại trước câu;
    DESTROY, MOVE -> đang tồn tại. Chỉ xét ô KHÔNG được nhắc (m_it = 0).
    B = 'trong câu t có thực thể j != i được nhắc và nhận nhãn B'."""
    cnt = defaultdict(Counter)
    for rec in records:
        n = len(rec["entities"])
        for t in range(1, len(rec["sentences"]) + 1):
            for i in range(n):
                if rec["mentions"][i][t - 1]:
                    continue
                y = rec["labels"][i][t - 1]
                e = rec["existence"][i][t - 1]
                for A in (CREATE, DESTROY, MOVE):
                    if (A == CREATE) == e:  # ô không hợp lệ cho nhãn A
                        continue
                    for B in (CREATE, DESTROY, MOVE):
                        sig = any(j != i and rec["mentions"][j][t - 1] and rec["labels"][j][t - 1] == B
                                  for j in range(n))
                        cnt[(A, B)][("sig" if sig else "nosig", y == A)] += 1
    res = {}
    for (A, B), c in cnt.items():
        a1, n1 = c[("sig", True)], c[("sig", True)] + c[("sig", False)]
        a0, n0 = c[("nosig", True)], c[("nosig", True)] + c[("nosig", False)]
        p1 = a1 / n1 if n1 else float("nan")
        p0 = a0 / n0 if n0 else float("nan")
        res[(A, B)] = {"lift": (p1 / p0) if p0 and p0 == p0 else float("nan"),
                       "n_implicit_A_with_B": a1, "p_sig": p1, "p_nosig": p0}
    return res


# ----------------------------------------------------------------------------- nạp dữ liệu
def load_jsonl(path: str) -> List[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_topics_from_naacl_grid(path: str) -> Dict[str, str]:
    """gold-full-grids.v3.tsv: dòng có cột 3 bắt đầu 'PROMPT:' cho biết chủ đề quy trình."""
    topics = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            c = line.rstrip("\n").split("\t")
            if len(c) > 2 and c[2].startswith("PROMPT:"):
                topics[c[0].strip()] = c[2][len("PROMPT:"):].strip()
    return topics


def load_action_file(path: str) -> List[Tuple[str, int, str, str, str, str]]:
    """answers.tsv của leaderboard: (pid, step, participant, action, before, after).
    Trả về DANH SÁCH (không dùng dict) để không âm thầm gộp các thực thể trùng tên."""
    out = []
    with open(path, encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            c = line.rstrip("\n\r").split("\t")
            if len(c) < 6:
                raise ValueError(f"{path}:{ln}: cần 6 cột, có {len(c)}")
            out.append((c[0], int(c[1]), c[2], c[3], c[4], c[5]))
    return out


def compare_action_rows(official: List[tuple], derived: List[tuple]) -> Dict[str, object]:
    """So sánh dạng đa tập theo (pid, step): bỏ qua thứ tự dòng, không gộp trùng tên."""
    a, b = defaultdict(list), defaultdict(list)
    for r in official:
        a[(str(r[0]), int(r[1]))].append(tuple(r[2:6]))
    for r in derived:
        b[(str(r[0]), int(r[1]))].append(tuple(r[2:6]))
    keys = set(a) | set(b)
    bad = sorted(k for k in keys if sorted(a[k]) != sorted(b[k]))
    return {"n_official_rows": len(official), "n_derived_rows": len(derived),
            "n_mismatched_steps": len(bad), "examples": bad[:5]}


# ----------------------------------------------------------------------------- chia chéo & mẫu kiểm tra
def stable_hash(s: str) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)


def cv_folds_by_topic(records: List[dict], k: int = 5, seed: int = 2026) -> Dict[str, int]:
    """Chia k phần theo CHỦ ĐỀ (prompt), tất định. Cân bằng tham lam đồng thời
    số điểm quyết định ngầm, số cặp vàng và số đoạn (mỗi đại lượng chuẩn hoá theo tổng).
    Trả về {paragraph_id: fold}."""
    by_topic = defaultdict(list)
    for r in records:
        by_topic[r.get("topic") or f"__no_topic__{r['paragraph_id']}"].append(r)
    stats = []
    for tp, rs in by_topic.items():
        v = (sum(len(decision_points(r)) for r in rs), sum(len(gold_pairs_report(r)) for r in rs), len(rs))
        stats.append((v, stable_hash(f"{seed}:{tp}"), tp))
    tot = [max(sum(s[0][d] for s in stats), 1) for d in range(3)]
    stats.sort(key=lambda x: (-sum(x[0][d] / tot[d] for d in range(3)), x[1]))
    load = [[0, 0, 0] for _ in range(k)]
    topic_fold = {}
    for v, _, tp in stats:
        f = min(range(k), key=lambda j: (max((load[j][d] + v[d]) / tot[d] for d in range(3)),
                                         sum(load[j][d] / tot[d] for d in range(3)), j))
        topic_fold[tp] = f
        for d in range(3):
            load[f][d] += v[d]
    return {r["paragraph_id"]: topic_fold[r.get("topic") or f"__no_topic__{r['paragraph_id']}"] for r in records}


def gold_audit_sample(pairs: List[dict], quotas: Dict[str, int], seed: int = 2026) -> List[dict]:
    """Lấy mẫu có chủ đích, tất định, cho người kiểm tra quan hệ X->Y.
    Nhóm (ưu tiên theo thứ tự): source_implicit, target_implicit, multi_pair, regular."""
    def group(p):
        if p["source_implicit"]:
            return "source_implicit"
        if p["target_implicit"]:
            return "target_implicit"
        if p["n_destroy_in_sentence"] * p["n_create_in_sentence"] > 1:
            return "multi_pair"
        return "regular"
    out = []
    for g, q in quotas.items():
        pool = [p for p in pairs if group(p) == g]
        pool.sort(key=lambda p: stable_hash(f"{seed}:{p['paragraph_id']}:{p['sentence_id']}:{p['source_idx']}:{p['target_idx']}"))
        for p in pool[:q]:
            out.append(dict(p, audit_group=g))
    return out
'''

TESTS_SRC = r'''
"""Kiểm thử đơn vị cho propara_core (chạy được bằng pytest hoặc run_all()).

Dữ liệu ở đây là ví dụ tự dựng (không phải mẫu ProPara thật).
"""
import propara_core as pc


def _toy(states, sents=None, parts=None, pid="toy"):
    n, T = len(states), len(states[0]) - 1
    raw = {"para_id": pid,
           "sentence_texts": sents or [f"Sentence {t}." for t in range(1, T + 1)],
           "participants": parts or [f"entity{i}" for i in range(n)],
           "states": states}
    return pc.build_record(raw, "train", topic="toy topic")


def test_derive_label_all_cases():
    assert pc.derive_label("leaf", "leaf") == pc.NONE
    assert pc.derive_label("-", "-") == pc.NONE
    assert pc.derive_label("?", "?") == pc.NONE
    assert pc.derive_label("-", "leaf") == pc.CREATE
    assert pc.derive_label("-", "?") == pc.CREATE          # tạo ra nhưng không rõ vị trí
    assert pc.derive_label("leaf", "-") == pc.DESTROY
    assert pc.derive_label("?", "-") == pc.DESTROY          # '?' vẫn là tồn tại
    assert pc.derive_label("leaf", "root") == pc.MOVE
    assert pc.derive_label("?", "leaf") == pc.MOVE          # quy ước báo cáo: còn lại là MOVE
    assert pc.derive_label("leaf", "?") == pc.MOVE


def test_exists_question_mark_is_existing():
    assert pc.exists("?") and pc.exists("leaf") and not pc.exists("-")


def test_mention_report_mode_singular_plural():
    assert pc.is_mentioned("plants", "The plant dies.")
    assert pc.is_mentioned("plant", "Plants die.")
    assert pc.is_mentioned("carbon dioxide; CO2", "The leaf absorbs co2.")   # tên gọi khác
    assert not pc.is_mentioned("the water", "The sun shines.")                # 'the' là từ rỗng
    assert not pc.is_mentioned("water", "Waterfalls are big.")                # khớp nguyên từ
    # chế độ report KHÔNG bắt -es / -ies; chế độ extended thì có
    assert not pc.is_mentioned("echo", "Echoes are produced.")
    assert pc.is_mentioned("echo", "Echoes are produced.", mode="extended")
    assert pc.is_mentioned("berry", "Berries grow.", mode="extended")


def test_build_record_labels_and_implicit():
    rec = _toy([["leaf", "leaf", "-"], ["-", "-", "leaf"]],
               sents=["The entity0 sits.", "Sugar is made."], parts=["entity0", "sugar"])
    assert rec["labels"] == [[pc.NONE, pc.DESTROY], [pc.NONE, pc.CREATE]]
    assert rec["implicit"][0] == [False, True]    # entity0 bị huỷ ở câu 2 mà không được nhắc
    assert rec["implicit"][1] == [False, False]   # sugar được nhắc
    assert pc.validate_record(rec) == []


def test_gold_pairs_cross_product_and_direction():
    rec = _toy([["a", "-"], ["b", "-"], ["-", "c"], ["-", "d"], ["e", "e"]])
    pairs = pc.gold_pairs_report(rec)
    got = {(p["source_idx"], p["target_idx"], p["sentence_id"]) for p in pairs}
    assert got == {(0, 2, 1), (0, 3, 1), (1, 2, 1), (1, 3, 1)}       # 2 x 2 = 4 cặp
    assert (2, 0, 1) not in got                                       # đúng chiều nguồn -> đích
    assert all(p["pair_gold_type"] == "derived_from_states" for p in pairs)


def test_no_pairs_when_only_destroy():
    rec = _toy([["a", "-"], ["b", "b"]])
    assert pc.gold_pairs_report(rec) == []


def test_official_conversion_same_step_and_next_step_rule():
    # câu 1: huỷ e0, không tạo; câu 2: chỉ tạo e1 -> rule b gán conversion về câu 1
    rec = _toy([["x", "-", "-", "-", "-"], ["-", "-", "y", "y", "y"]])
    conv = pc.official_q3_conversions(rec)
    assert len(conv) == 1 and conv[0]["step_id"] == 1 and conv[0]["rule"] == "next_step_create"
    # cùng câu: một conversion gộp mọi D và C
    rec2 = _toy([["a", "-"], ["b", "-"], ["-", "c"]])
    conv2 = pc.official_q3_conversions(rec2)
    assert len(conv2) == 1 and conv2[0]["destroyed"] == "entity0 AND entity1"
    assert len(pc.gold_pairs_report(rec2)) == 2     # trong khi định nghĩa cặp của báo cáo cho 2 cặp


def test_official_conversion_keeps_evaluator_off_by_one():
    # T=3: huỷ ở câu 2, tạo ở câu 3. Evaluator chỉ xét rule b khi step < T-1 = 2 -> KHÔNG có conversion
    rec = _toy([["x", "x", "-", "-"], ["-", "-", "-", "y"]])
    assert pc.official_q3_conversions(rec) == []


def test_decision_points_definition():
    # câu 1: e0 bị huỷ ngầm; ứng viên DESTROY = các thực thể đang tồn tại (e0, e1) -> 1 điểm quyết định
    rec = _toy([["a", "-"], ["b", "b"], ["-", "-"]],
               sents=["Something happens."], parts=["alpha", "beta", "gamma"])
    dps = pc.decision_points(rec)
    assert len(dps) == 1 and dps[0]["label"] == pc.DESTROY
    assert dps[0]["candidates"] == [0, 1] and dps[0]["gold"] == [0]
    assert abs(dps[0]["random_acc"] - 0.5) < 1e-12


def test_decision_point_excludes_mentioned_gold_and_needs_two_candidates():
    # beta bị huỷ và ĐƯỢC NHẮC -> bị loại khỏi ứng viên; còn alpha (ngầm) + gamma
    rec = _toy([["a", "-"], ["b", "-"], ["c", "c"]],
               sents=["The beta breaks."], parts=["alpha", "beta", "gamma"])
    dps = pc.decision_points(rec)
    assert len(dps) == 1 and dps[0]["candidates"] == [0, 2]
    # chỉ còn 1 ứng viên -> không phải điểm quyết định
    rec2 = _toy([["a", "-"], ["b", "-"]], sents=["The beta breaks."], parts=["alpha", "beta"])
    assert pc.decision_points(rec2) == []


def test_constraint_violations_self_evolving():
    # DESTROY khi chưa tồn tại, rồi CREATE hợp lệ
    bad, tot = pc.count_constraint_violations([[pc.DESTROY, pc.CREATE]], [False])
    assert (bad, tot) == (1, 2)
    # DESTROY hai lần liên tiếp mà không CREATE lại -> lần 2 vi phạm
    bad, _ = pc.count_constraint_violations([[pc.DESTROY, pc.DESTROY]], [True])
    assert bad == 1
    # chuỗi hợp lệ
    bad, _ = pc.count_constraint_violations([[pc.NONE, pc.CREATE, pc.MOVE, pc.DESTROY]], [False])
    assert bad == 0


def test_gold_sequences_never_violate_constraints():
    rec = _toy([["a", "-", "-", "b", "?"], ["-", "x", "y", "-", "-"]])
    init = [rec["existence"][i][0] for i in range(2)]
    assert pc.count_constraint_violations(rec["labels"], init)[0] == 0


def test_validate_record_catches_errors():
    raw = {"para_id": "bad", "sentence_texts": ["One.", ""], "participants": ["a", "A"],
           "states": [["x", "x", "x"], ["x", ""]]}
    issues = pc.validate_record(pc.build_record(raw, "dev"))
    text = " | ".join(issues)
    assert "câu 2 rỗng" in text and "trùng tên" in text and "cần 3" in text


def test_pp2_counts_gold_ceiling():
    rec = _toy([["a", "-"], ["-", "c"], ["e", "e"]])
    c = pc.pp2_counts(rec)
    assert c["all_pairs"] == 1 * 3 * 2
    assert c["kept_gold_existence"] == 2       # (e0->e1), (e2->e1)
    assert c["gold_kept"] == c["gold_pairs"] == 1


def test_compare_action_rows_ignores_order_but_not_content():
    a = [("1", 1, "x", "NONE", "-", "-"), ("1", 1, "y", "CREATE", "-", "leaf")]
    assert pc.compare_action_rows(a, list(reversed(a)))["n_mismatched_steps"] == 0
    b = [("1", 1, "x", "NONE", "-", "-"), ("1", 1, "y", "CREATE", "-", "root")]
    assert pc.compare_action_rows(a, b)["n_mismatched_steps"] == 1


def test_cv_folds_deterministic_and_grouped():
    recs = [_toy([["a", "-"], ["-", "b"]], pid=str(i)) for i in range(10)]
    for i, r in enumerate(recs):
        r["topic"] = f"topic{i % 4}"
    f1 = pc.cv_folds_by_topic(recs, k=3)
    assert f1 == pc.cv_folds_by_topic(recs, k=3)
    for t in range(4):
        assert len({f1[r["paragraph_id"]] for r in recs if r["topic"] == f"topic{t}"}) == 1


def run_all(verbose=True):
    names = sorted(n for n in globals() if n.startswith("test_"))
    failed = []
    for n in names:
        try:
            globals()[n]()
            if verbose:
                print(f"  PASS  {n}")
        except Exception as e:  # noqa: BLE001 — báo lỗi rõ ràng rồi tổng hợp
            failed.append((n, repr(e)))
            print(f"  FAIL  {n}: {e!r}")
    return len(names), failed
'''

(OUT / "propara_core.py").write_text(CORE_SRC, encoding="utf-8")
(OUT / "propara_core_tests.py").write_text(TESTS_SRC, encoding="utf-8")
sys.dont_write_bytecode = True   # không để __pycache__ lọt vào output
sys.path.insert(0, str(OUT))
import importlib
import propara_core as pc
import propara_core_tests as pct
importlib.reload(pc); importlib.reload(pct)
print("propara_core", pc.CORE_VERSION, "| sha256", hashlib.sha256(CORE_SRC.encode()).hexdigest()[:16])

propara_core core_v1 | sha256 7cab85b1184a58f9


## 4. Kiểm thử đơn vị (phải đạt hết trước khi đụng vào dữ liệu)

In [4]:
n_tests, failed_tests = pct.run_all(verbose=True)
print(f"\n{n_tests - len(failed_tests)}/{n_tests} kiểm thử đạt")
if failed_tests:
    raise RuntimeError(f"Kiểm thử đơn vị thất bại: {failed_tests}")

  PASS  test_build_record_labels_and_implicit
  PASS  test_compare_action_rows_ignores_order_but_not_content
  PASS  test_constraint_violations_self_evolving
  PASS  test_cv_folds_deterministic_and_grouped
  PASS  test_decision_point_excludes_mentioned_gold_and_needs_two_candidates
  PASS  test_decision_points_definition
  PASS  test_derive_label_all_cases
  PASS  test_exists_question_mark_is_existing
  PASS  test_gold_pairs_cross_product_and_direction
  PASS  test_gold_sequences_never_violate_constraints
  PASS  test_mention_report_mode_singular_plural
  PASS  test_no_pairs_when_only_destroy
  PASS  test_official_conversion_keeps_evaluator_off_by_one
  PASS  test_official_conversion_same_step_and_next_step_rule
  PASS  test_pp2_counts_gold_ceiling
  PASS  test_validate_record_catches_errors

16/16 kiểm thử đạt


## 5. Nạp dữ liệu gốc: dò `/kaggle/input`, nếu không có thì tải theo commit ghim; kiểm tra SHA-256

In [5]:
def find_in_input(suffixes):
    if not INPUT_ROOT.exists():
        return None
    for suf in suffixes:
        base = suf.split("/")[-1]
        for p in sorted(INPUT_ROOT.rglob(base)):
            if p.is_file() and p.as_posix().endswith(suf):
                return p
    return None

RAW, RAW_SOURCE, CHECKSUM = {}, {}, {}
for name, (url, sha, suffixes) in RAW_FILES.items():
    p = find_in_input(suffixes)
    if p is not None:
        RAW_SOURCE[name] = f"input:{p}"
    else:
        p = OUT / "raw" / name
        if not p.exists():
            if not ALLOW_DOWNLOAD or ENV["internet"] is not True:
                raise FileNotFoundError(
                    f"Không tìm thấy {name} trong {INPUT_ROOT} và không tải được (internet={ENV['internet']}).\n"
                    "Cách sửa: bật Settings → Internet → On, HOẶC Add Input dataset chứa propara-raw-v1.zip đã giải nén.")
            for attempt in range(3):
                try:
                    urllib.request.urlretrieve(url, p)
                    break
                except Exception as e:
                    if attempt == 2:
                        raise RuntimeError(f"Tải thất bại {url}: {e}")
                    time.sleep(2)
        RAW_SOURCE[name] = f"download:{url}"
    RAW[name] = p
    got = pc.sha256_file(str(p))
    CHECKSUM[name] = {"expected": sha, "observed": got, "match": got == sha}
    print(f"{'OK ' if got == sha else 'LỆCH'}  {name:32s} {RAW_SOURCE[name][:90]}")

OK   grids.v1.train.json              input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\grids.v1.train.j
OK   grids.v1.dev.json                input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\grids.v1.dev.jso
OK   grids.v1.test.json               input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\grids.v1.test.js
OK   naacl18.partition.paraid.tsv     input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\naacl18.partitio
OK   gold-full-grids.v3.tsv           input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\gold-full-grids.
OK   leaderboard_answers_train.tsv    input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\leaderboard_answ
OK   leaderboard_answers_dev.tsv      input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\propara-raw-v1\leaderboard_answ
OK   leaderboard_answers_test.tsv     input:C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\fakeA\input\

## 6. Chuẩn hoá và kiểm tra cấu trúc

In [6]:
TOPICS = pc.load_topics_from_naacl_grid(str(RAW["gold-full-grids.v3.tsv"]))
PARTITION = {}
with open(RAW["naacl18.partition.paraid.tsv"], encoding="utf-8") as f:
    for line in f:
        if line.strip():
            sp, pid = line.split("\t")[:2]
            PARTITION[pid.strip()] = sp.strip()

RAWS = {sp: pc.load_jsonl(str(RAW[f"grids.v1.{sp}.json"])) for sp in SPLITS}
if SMOKE_TEST:
    RAWS = {sp: rows[:15] for sp, rows in RAWS.items()}
RECS = [pc.build_record(r, sp, TOPICS.get(str(r["para_id"]))) for sp in SPLITS for r in RAWS[sp]]
BY_SPLIT = {sp: [r for r in RECS if r["split"] == sp] for sp in SPLITS}

# --- kiểm tra cấu trúc từng đoạn
ISSUES = [(r["paragraph_id"], msg) for r in RECS for msg in pc.validate_record(r)]
DUP_ISSUES = [x for x in ISSUES if "trùng tên" in x[1]]
HARD_ISSUES = [x for x in ISSUES if "trùng tên" not in x[1]]

# --- trùng lặp / rò rỉ giữa các tập
pid_split = defaultdict(set)
for r in RECS:
    pid_split[r["paragraph_id"]].add(r["split"])
PID_DUP_WITHIN = [p for p, c in Counter(r["paragraph_id"] for r in RECS).items() if c > 1]
PID_CROSS = [p for p, s in pid_split.items() if len(s) > 1]
PARTITION_MISMATCH = [r["paragraph_id"] for r in RECS if PARTITION.get(r["paragraph_id"]) != r["split"]]
topic_split = defaultdict(set)
for r in RECS:
    topic_split[r["topic"]].add(r["split"])
TOPIC_CROSS = [t for t, s in topic_split.items() if len(s) > 1]
NO_TOPIC = [r["paragraph_id"] for r in RECS if not r["topic"]]
text_split = defaultdict(set)
sent_split = defaultdict(set)
for r in RECS:
    text_split[" ".join(r["sentences"]).strip().lower()].add(r["split"])
    for s in r["sentences"]:
        sent_split[s.strip().lower()].add(r["split"])
TEXT_CROSS = [t for t, s in text_split.items() if len(s) > 1]
SENT_CROSS = sorted(t for t, s in sent_split.items() if len(s) > 1)

# --- ràng buộc tồn tại trên nhãn vàng (phải = 0 theo cấu tạo)
GOLD_VIOL = sum(pc.count_constraint_violations(r["labels"], [e[0] for e in r["existence"]])[0] for r in RECS)

COUNTS = {sp: {"paragraphs": len(BY_SPLIT[sp]),
               "sentences": sum(len(r["sentences"]) for r in BY_SPLIT[sp]),
               "entities": sum(len(r["entities"]) for r in BY_SPLIT[sp])} for sp in SPLITS}
COUNTS["all"] = {k: sum(COUNTS[sp][k] for sp in SPLITS) for k in ("paragraphs", "sentences", "entities")}
print(json.dumps(COUNTS, indent=1))
print("Lỗi cấu trúc nghiêm trọng:", len(HARD_ISSUES), HARD_ISSUES[:5])
print("Thực thể trùng tên:", DUP_ISSUES)
print("Trùng mã đoạn trong/giữa tập:", PID_DUP_WITHIN, PID_CROSS, "| lệch file phân chia:", PARTITION_MISMATCH[:5])
print("Chủ đề xuất hiện ở >1 tập:", len(TOPIC_CROSS), "| đoạn thiếu chủ đề:", len(NO_TOPIC), "| số chủ đề:", len(topic_split))
print("Đoạn văn trùng nguyên văn giữa các tập:", len(TEXT_CROSS), "| câu trùng nguyên văn giữa các tập:", len(SENT_CROSS))
print("Vi phạm ràng buộc tồn tại trên nhãn vàng:", GOLD_VIOL)

{
 "train": {
  "paragraphs": 391,
  "sentences": 2639,
  "entities": 1504
 },
 "dev": {
  "paragraphs": 43,
  "sentences": 290,
  "entities": 175
 },
 "test": {
  "paragraphs": 54,
  "sentences": 373,
  "entities": 236
 },
 "all": {
  "paragraphs": 488,
  "sentences": 3302,
  "entities": 1915
 }
}
Lỗi cấu trúc nghiêm trọng: 0 []
Thực thể trùng tên: [('263', "263: thực thể trùng tên 'bales'"), ('1151', "1151: thực thể trùng tên 'new element'"), ('1204', "1204: thực thể trùng tên 'mrna'"), ('1207', "1207: thực thể trùng tên 'skin'"), ('893', "893: thực thể trùng tên 'glucose'"), ('901', "901: thực thể trùng tên 'sound'")]
Trùng mã đoạn trong/giữa tập: [] [] | lệch file phân chia: []
Chủ đề xuất hiện ở >1 tập: 0 | đoạn thiếu chủ đề: 0 | số chủ đề: 183
Đoạn văn trùng nguyên văn giữa các tập: 0 | câu trùng nguyên văn giữa các tập: 18
Vi phạm ràng buộc tồn tại trên nhãn vàng: 0


## 7. Đối chiếu với đáp án của bộ chấm chính thức (`aristo-leaderboard/propara/data/*/answers.tsv`)

In [7]:
ANSWER_CMP = {}
for sp in SPLITS:
    official = pc.load_action_file(str(RAW[f"leaderboard_answers_{sp}.tsv"]))
    if SMOKE_TEST:
        keep = {r["paragraph_id"] for r in BY_SPLIT[sp]}
        official = [x for x in official if x[0] in keep]
    derived = [row for r in BY_SPLIT[sp] for row in pc.to_action_rows(r)]
    ANSWER_CMP[sp] = pc.compare_action_rows(official, derived)
    print(sp, ANSWER_CMP[sp])

train {'n_official_rows': 10356, 'n_derived_rows': 10356, 'n_mismatched_steps': 0, 'examples': []}
dev {'n_official_rows': 1204, 'n_derived_rows': 1204, 'n_mismatched_steps': 0, 'examples': []}
test {'n_official_rows': 1674, 'n_derived_rows': 1674, 'n_mismatched_steps': 0, 'examples': []}


## 8. Suy nhãn và thay đổi ngầm (đối chiếu Bảng 1)

In [8]:
TABLE1 = {sp: {} for sp in SPLITS}
for sp in SPLITS:
    for lab in ("CREATE", "DESTROY", "MOVE", "NONE"):
        tot = imp = 0
        for r in BY_SPLIT[sp]:
            for i in range(len(r["entities"])):
                for t, y in enumerate(r["labels"][i]):
                    if y == lab:
                        tot += 1
                        imp += int(r["implicit"][i][t])
        TABLE1[sp][lab] = (imp, tot)
print(f"{'tập':6s} {'nhãn':8s} {'ngầm/tổng (đo)':>16s} {'báo cáo':>12s}")
for sp in SPLITS:
    for lab in ("CREATE", "DESTROY", "MOVE"):
        imp, tot = TABLE1[sp][lab]
        rep = REPORT["table1"][sp][lab]
        print(f"{sp:6s} {lab:8s} {f'{imp}/{tot} ({imp/max(tot,1):.0%})':>16s} {f'{rep[0]}/{rep[1]}':>12s} {'✓' if (imp, tot) == rep else '✗'}")

tập    nhãn       ngầm/tổng (đo)      báo cáo


train  CREATE       78/747 (10%)       78/747 ✓
train  DESTROY      81/502 (16%)       81/502 ✓
train  MOVE       355/1211 (29%)     355/1211 ✓
dev    CREATE        11/85 (13%)        11/85 ✓
dev    DESTROY       15/68 (22%)        15/68 ✓
dev    MOVE         41/157 (26%)       41/157 ✓
test   CREATE       15/128 (12%)       15/128 ✓
test   DESTROY      22/121 (18%)       22/121 ✓
test   MOVE         53/152 (35%)       53/152 ✓


## 9. Cặp vàng (định nghĩa báo cáo) và conversion theo bộ chấm chính thức

In [9]:
PAIRS = [p for r in RECS for p in pc.gold_pairs_report(r)]
PAIRS_SPLIT = Counter(p["split"] for p in PAIRS)
OFFICIAL_CONV = [dict(c, paragraph_id=r["paragraph_id"], split=r["split"]) for r in RECS for c in pc.official_q3_conversions(r)]
OFFICIAL_SPLIT = Counter(c["split"] for c in OFFICIAL_CONV)
DUP_NAME_PIDS = sorted({pid for pid, _ in DUP_ISSUES})

shape = Counter()
for r in RECS:
    for t in range(1, len(r["sentences"]) + 1):
        D, C = pc.destroyed_created_at(r, t)
        if D and C:
            shape[(len(D), len(C))] += 1
PAIR_STATS = {
    "pairs_by_split": dict(PAIRS_SPLIT),
    "source_implicit": sum(p["source_implicit"] for p in PAIRS),
    "target_implicit": sum(p["target_implicit"] for p in PAIRS),
    "any_side_implicit": sum(p["source_implicit"] or p["target_implicit"] for p in PAIRS),
    "both_sides_implicit": sum(p["source_implicit"] and p["target_implicit"] for p in PAIRS),
    "sentences_with_pairs": sum(shape.values()),
    "sentences_1x1": shape[(1, 1)],
    "pairs_from_multi_entity_sentences": sum(p["n_destroy_in_sentence"] * p["n_create_in_sentence"] > 1 for p in PAIRS),
    "sentence_shapes_DxC": {f"{d}x{c}": v for (d, c), v in sorted(shape.items())},
    "official_q3_conversions_replica_by_split": dict(OFFICIAL_SPLIT),
    "official_q3_rules": dict(Counter(c["rule"] for c in OFFICIAL_CONV)),
}
for sp in SPLITS:
    PAIR_STATS[f"pair_slices_{sp}"] = {
        "n": PAIRS_SPLIT[sp],
        "source_implicit": sum(p["source_implicit"] for p in PAIRS if p["split"] == sp),
        "target_implicit": sum(p["target_implicit"] for p in PAIRS if p["split"] == sp),
        "any_side_implicit": sum(p["source_implicit"] or p["target_implicit"] for p in PAIRS if p["split"] == sp),
    }
print(json.dumps(PAIR_STATS, indent=1, ensure_ascii=False))
print("\nĐoạn có thực thể trùng tên NGUYÊN VĂN (evaluator chính thức gộp làm một):", DUP_NAME_PIDS)

{
 "pairs_by_split": {
  "train": 378,
  "dev": 58,
  "test": 97
 },
 "source_implicit": 106,
 "target_implicit": 92,
 "any_side_implicit": 178,
 "both_sides_implicit": 20,
 "sentences_with_pairs": 439,
 "sentences_1x1": 369,
 "pairs_from_multi_entity_sentences": 164,
 "sentence_shapes_DxC": {
  "1x1": 369,
  "1x2": 21,
  "1x4": 1,
  "2x1": 35,
  "2x2": 3,
  "3x1": 6,
  "3x2": 1,
  "4x1": 3
 },
 "official_q3_conversions_replica_by_split": {
  "train": 354,
  "dev": 49,
  "test": 82
 },
 "official_q3_rules": {
  "same_step": 439,
  "next_step_create": 34,
  "next_step_destroy": 12
 },
 "pair_slices_train": {
  "n": 378,
  "source_implicit": 73,
  "target_implicit": 65,
  "any_side_implicit": 125
 },
 "pair_slices_dev": {
  "n": 58,
  "source_implicit": 16,
  "target_implicit": 11,
  "any_side_implicit": 21
 },
 "pair_slices_test": {
  "n": 97,
  "source_implicit": 17,
  "target_implicit": 16,
  "any_side_implicit": 32
 }
}

Đoạn có thực thể trùng tên NGUYÊN VĂN (evaluator chính thức gộp

## 10. Điểm quyết định ngầm, PP2, lọc đồng xuất hiện, lift (Bảng 2)

In [10]:
DPS = [d for r in RECS for d in pc.decision_points(r)]
DP_STATS = {}
for lab in ("DESTROY", "CREATE"):
    x = [d for d in DPS if d["label"] == lab]
    DP_STATS[lab] = {"n": len(x),
                     "by_split": dict(Counter(d["split"] for d in x)),
                     "avg_candidates": round(statistics.mean(d["n_candidates"] for d in x), 3) if x else None,
                     "random_acc": round(statistics.mean(d["random_acc"] for d in x), 4) if x else None}
PP2 = Counter()
for r in RECS:
    PP2.update(pc.pp2_counts(r))
COOC = Counter()
for r in RECS:
    COOC.update(pc.cooccurrence_filter_coverage(r))
COOC_PCT = {
    "same_sentence_gold": 100 * COOC["same_sentence"] / max(COOC["gold"], 1),
    "same_sentence_src_implicit": 100 * COOC["same_sentence_src_implicit"] / max(COOC["gold_src_implicit"], 1),
    "any_sentence_gold": 100 * COOC["any_sentence"] / max(COOC["gold"], 1),
    "any_sentence_src_implicit": 100 * COOC["any_sentence_src_implicit"] / max(COOC["gold_src_implicit"], 1),
}
LIFT = pc.lift_table(RECS)
print("Điểm quyết định:", json.dumps(DP_STATS, ensure_ascii=False))
print("PP1/PP2:", dict(PP2), f"| tỉ lệ giữ = {PP2['kept_gold_existence']/max(PP2['all_pairs'],1):.1%}")
print("Lọc đồng xuất hiện (%):", {k: round(v, 1) for k, v in COOC_PCT.items()})
print("\nLift (đo | báo cáo):")
for A in ("CREATE", "DESTROY", "MOVE"):
    row = []
    for B in ("CREATE", "DESTROY", "MOVE"):
        v = LIFT.get((A, B), {"lift": float("nan"), "n_implicit_A_with_B": 0})
        rep = REPORT["lift"][(A, B)]
        row.append(f"{v['lift']:.1f}({v['n_implicit_A_with_B']}) | {rep[0]}({rep[1]})")
    print(f"  {A:8s}", "   ".join(row))

Điểm quyết định: {"DESTROY": {"n": 57, "by_split": {"train": 37, "dev": 10, "test": 10}, "avg_candidates": 2.649, "random_acc": 0.5058}, "CREATE": {"n": 76, "by_split": {"train": 56, "dev": 8, "test": 12}, "avg_candidates": 2.947, "random_acc": 0.3981}}
PP1/PP2: {'all_pairs': 48070, 'kept_gold_existence': 11144, 'gold_pairs': 533, 'gold_kept': 533, 'label_questions': 52936} | tỉ lệ giữ = 23.2%
Lọc đồng xuất hiện (%): {'same_sentence_gold': 66.6, 'same_sentence_src_implicit': 0.0, 'any_sentence_gold': 74.7, 'any_sentence_src_implicit': 18.9}

Lift (đo | báo cáo):
  CREATE   0.3(10) | 0.3(10)   6.6(63) | 6.6(63)   0.3(10) | 0.3(10)
  DESTROY  9.0(85) | 9.0(85)   1.4(19) | 1.4(19)   0.0(2) | 0.0(2)
  MOVE     1.0(97) | 1.0(97)   1.3(69) | 1.3(69)   1.2(141) | 1.2(141)


## 11. Độ nhạy của cờ nhắc và khả thi của PP4 (giải mã chính xác)

In [11]:
# (a) Cờ nhắc kiểu 'extended' (bắt thêm -es/-ies) thay đổi bao nhiêu ô ngầm? Chỉ để ghi nhận độ nhạy.
SENS = Counter()
for sp in SPLITS:
    for raw in RAWS[sp]:
        rx = pc.build_record(raw, sp, mention_mode="extended")
        rr = next(r for r in BY_SPLIT[sp] if r["paragraph_id"] == str(raw["para_id"]))
        for i in range(len(rr["entities"])):
            for t in range(len(rr["labels"][i])):
                if rr["implicit"][i][t] != rx["implicit"][i][t]:
                    SENS[(sp, rr["labels"][i][t])] += 1
SENS_D = {f"{sp}/{lab}": v for (sp, lab), v in sorted(SENS.items())}
print("Số ô ngầm thay đổi khi dùng cờ nhắc 'extended':", SENS_D or "0")

# (b) Số thực thể/đoạn → kích thước không gian giải mã chính xác 4^n (cận trên trong báo cáo, mục 4.4)
NS = [len(r["entities"]) for r in RECS]
TS = [len(r["sentences"]) for r in RECS]
PP4 = {"mean_entities": round(statistics.mean(NS), 3), "max_entities": max(NS),
       "entities_hist": dict(sorted(Counter(NS).items())), "mean_sentences": round(statistics.mean(TS), 3),
       "max_sentences": max(TS), "max_states_per_step_4^n": 4 ** max(NS),
       "max_T_x_4^n": max(len(r["sentences"]) * 4 ** len(r["entities"]) for r in RECS),
       "paragraphs_n_gt_8": sum(n > 8 for n in NS)}
print(json.dumps(PP4, indent=1))

Số ô ngầm thay đổi khi dùng cờ nhắc 'extended': {'train/CREATE': 2, 'train/DESTROY': 1, 'train/MOVE': 1}
{
 "mean_entities": 3.924,
 "max_entities": 8,
 "entities_hist": {
  "1": 6,
  "2": 93,
  "3": 117,
  "4": 111,
  "5": 83,
  "6": 42,
  "7": 27,
  "8": 9
 },
 "mean_sentences": 6.766,
 "max_sentences": 10,
 "max_states_per_step_4^n": 65536,
 "max_T_x_4^n": 655360,
 "paragraphs_n_gt_8": 0
}


## 12. Chia chéo 5 phần theo chủ đề (khoá trước) và mẫu kiểm tra nhãn vàng cho người chấm

In [12]:
FOLDS = pc.cv_folds_by_topic(RECS, k=5, seed=SEED)
FOLD_STATS = defaultdict(Counter)
for r in RECS:
    f = FOLDS[r["paragraph_id"]]
    FOLD_STATS[f]["paragraphs"] += 1
    FOLD_STATS[f]["gold_pairs"] += len(pc.gold_pairs_report(r))
    for d in pc.decision_points(r):
        FOLD_STATS[f][f"dp_{d['label']}"] += 1
for f in sorted(FOLD_STATS):
    print("fold", f, dict(FOLD_STATS[f]))

AUDIT_QUOTAS = {"source_implicit": 30, "target_implicit": 20, "multi_pair": 20, "regular": 10}
AUDIT = pc.gold_audit_sample(PAIRS, AUDIT_QUOTAS, seed=SEED)
print("Mẫu kiểm tra nhãn vàng:", dict(Counter(a["audit_group"] for a in AUDIT)), "| theo tập:", dict(Counter(a["split"] for a in AUDIT)))

fold

 0 {'paragraphs': 99, 'gold_pairs': 103, 'dp_DESTROY': 12, 'dp_CREATE': 15}
fold 1 {'paragraphs': 95, 'gold_pairs': 108, 'dp_DESTROY': 10, 'dp_CREATE': 17}
fold 2 {'paragraphs': 99, 'gold_pairs': 108, 'dp_CREATE': 14, 'dp_DESTROY': 12}
fold 3 {'paragraphs': 99, 'gold_pairs': 108, 'dp_CREATE': 15, 'dp_DESTROY': 11}
fold 4 {'paragraphs': 96, 'gold_pairs': 106, 'dp_CREATE': 15, 'dp_DESTROY': 12}
Mẫu kiểm tra nhãn vàng: {'source_implicit': 30, 'target_implicit': 20, 'multi_pair': 20, 'regular': 10} | theo tập: {'train': 57, 'test': 14, 'dev': 9}


## 13. Biểu đồ

In [13]:
FIGS = []
if ENV["matplotlib"]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    C3 = {"train": "#2a78d6", "dev": "#eb6834", "test": "#1baf7a"}   # 3 màu phân loại đầu, cố định theo tập
    INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

    def _style(ax):
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(MUTED)
        ax.tick_params(colors=MUTED, length=0)
        ax.yaxis.grid(True, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)

    # Hình 1: tỉ lệ thay đổi ngầm theo nhãn và tập
    fig, ax = plt.subplots(figsize=(8, 4.2), dpi=130)
    labs = ["CREATE", "DESTROY", "MOVE"]
    w = 0.26
    for k, sp in enumerate(SPLITS):
        vals = [100 * TABLE1[sp][l][0] / max(TABLE1[sp][l][1], 1) for l in labs]
        xs = [j + (k - 1) * (w + 0.02) for j in range(len(labs))]
        bars = ax.bar(xs, vals, width=w, color=C3[sp], label=sp, edgecolor="white", linewidth=2)
        for x, v, l in zip(xs, vals, labs):
            ax.text(x, v + 0.8, f"{TABLE1[sp][l][0]}/{TABLE1[sp][l][1]}", ha="center", va="bottom", fontsize=7, color=MUTED)
    ymax = max(100 * TABLE1[sp][l][0] / max(TABLE1[sp][l][1], 1) for sp in SPLITS for l in labs)
    ax.set_ylim(0, ymax * 1.22)
    ax.set_xticks(range(len(labs)), labs, color=INK)
    ax.set_ylabel("% thay đổi là ngầm", color=MUTED)
    ax.set_title("Thay đổi ngầm chiếm 10–35% số thay đổi; CREATE/DESTROY ngầm là nhóm nhỏ", loc="left", fontsize=10, color=INK)
    ax.legend(frameon=False, ncol=3, loc="upper left")
    _style(ax)
    fig.tight_layout()
    p = OUT / "reports/figures/fig1_implicit_rate_by_label.png"
    fig.savefig(p); FIGS.append(p.name); plt.show(); plt.close(fig)

    # Hình 2: số thực thể mỗi đoạn (quyết định chi phí giải mã chính xác của PP4)
    fig, ax = plt.subplots(figsize=(8, 3.8), dpi=130)
    hist = Counter(NS)
    xs = sorted(hist)
    ax.bar(xs, [hist[x] for x in xs], width=0.7, color="#2a78d6", edgecolor="white", linewidth=2)
    for x in xs:
        ax.text(x, hist[x] + 1, str(hist[x]), ha="center", va="bottom", fontsize=8, color=MUTED)
    ax.set_ylim(0, max(hist.values()) * 1.15)
    ax.set_xticks(xs)
    ax.set_xlabel("số thực thể n trong đoạn", color=MUTED)
    ax.set_ylabel("số đoạn", color=MUTED)
    ax.set_title(f"n tối đa = {max(NS)} → 4^n ≤ {4 ** max(NS):,} cấu hình/câu: giải mã chính xác khả thi", loc="left", fontsize=10, color=INK)
    _style(ax)
    fig.tight_layout()
    p = OUT / "reports/figures/fig2_entities_per_paragraph.png"
    fig.savefig(p); FIGS.append(p.name); plt.show(); plt.close(fig)
else:
    print("Không có matplotlib — bỏ qua biểu đồ (không ảnh hưởng cổng nghiệm thu).")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14324\2314551064.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(p); FIGS.append(p.name); plt.show(); plt.close(fig)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14324\2314551064.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(p); FIGS.append(p.name); plt.show(); plt.close(fig)


## 14. Cổng nghiệm thu: đối chiếu số liệu báo cáo + kiểm tra cấu trúc

In [14]:
CHECKS = []
def check(cid, desc, expected, observed, ok, severity="FAIL", note=""):
    status = "PASS" if ok else severity
    if SMOKE_TEST and cid.startswith("R"):
        status = "SKIP(smoke)"
    CHECKS.append({"id": cid, "check": desc, "expected": str(expected), "observed": str(observed), "status": status, "note": note})

# --- S: cấu trúc, toàn vẹn, rò rỉ
check("S01", "Kiểm thử đơn vị", f"{n_tests}/{n_tests}", f"{n_tests - len(failed_tests)}/{n_tests}", not failed_tests)
check("S02", "SHA-256 file gốc khớp giá trị ghim", "8/8", f"{sum(v['match'] for v in CHECKSUM.values())}/8",
      all(v["match"] for v in CHECKSUM.values()), "WARN", "lệch = bản dữ liệu khác; các kiểm tra R quyết định")
check("S03", "Không có lỗi cấu trúc (độ dài trạng thái, câu rỗng, trạng thái rỗng)", 0, len(HARD_ISSUES), not HARD_ISSUES)
check("S04", "Thực thể trùng tên trong cùng đoạn", 0, len(DUP_ISSUES), not DUP_ISSUES, "WARN",
      f"đoạn {sorted({p for p, _ in DUP_ISSUES})}; giữ nguyên, định danh thực thể bằng chỉ số")
check("S05", "Mã đoạn duy nhất, không chéo tập, khớp file phân chia", "0/0/0",
      f"{len(PID_DUP_WITHIN)}/{len(PID_CROSS)}/{len(PARTITION_MISMATCH)}", not (PID_DUP_WITHIN or PID_CROSS or PARTITION_MISMATCH))
check("S06", "Không có chủ đề (prompt) nằm ở >1 tập", 0, len(TOPIC_CROSS), not TOPIC_CROSS, "WARN")
check("S07", "Không có đoạn văn trùng nguyên văn giữa các tập", 0, len(TEXT_CROSS), not TEXT_CROSS)
check("S08", "Câu trùng nguyên văn giữa các tập (thông tin)", "ghi nhận", len(SENT_CROSS), True, "INFO",
      "câu ngắn chung chung kiểu 'the magma cools.'; không phải rò rỉ nhãn")
check("S09", "Nhãn vàng không vi phạm ràng buộc tồn tại", 0, GOLD_VIOL, GOLD_VIOL == 0)
ans_bad = {sp: ANSWER_CMP[sp]["n_mismatched_steps"] for sp in SPLITS}
check("S10", "Hành động suy từ grids.v1 = answers.tsv chính thức (so đa tập)", "0/0/0",
      "/".join(str(ans_bad[sp]) for sp in SPLITS), sum(ans_bad.values()) == 0)
check("S11", "Mọi đoạn có chủ đề (để chia chéo theo chủ đề)", 0, len(NO_TOPIC), not NO_TOPIC, "WARN")

# --- R: con số trong báo cáo
c = COUNTS["all"]
check("R01", "Số đoạn / câu / thực thể (tr.1)", "488/3302/1915", f"{c['paragraphs']}/{c['sentences']}/{c['entities']}",
      (c["paragraphs"], c["sentences"], c["entities"]) == (488, 3302, 1915))
check("R02", "Số đoạn train/dev/test (file phân chia)", "391/43/54", "/".join(str(COUNTS[sp]["paragraphs"]) for sp in SPLITS),
      all(COUNTS[sp]["paragraphs"] == EXPECTED_SPLIT_PARAS[sp] for sp in SPLITS))
t1_tot_ok = all(TABLE1[sp][l][1] == REPORT["table1"][sp][l][1] for sp in SPLITS for l in ("CREATE", "DESTROY", "MOVE"))
t1_imp_ok = all(TABLE1[sp][l][0] == REPORT["table1"][sp][l][0] for sp in SPLITS for l in ("CREATE", "DESTROY", "MOVE"))
check("R03", "Bảng 1: tổng số nhãn theo tập (9 ô)", "khớp 9/9",
      f"khớp {sum(TABLE1[sp][l][1] == REPORT['table1'][sp][l][1] for sp in SPLITS for l in ('CREATE','DESTROY','MOVE'))}/9", t1_tot_ok)
check("R04", "Bảng 1: số thay đổi ngầm (9 ô, cờ nhắc kiểu báo cáo)", "khớp 9/9",
      f"khớp {sum(TABLE1[sp][l][0] == REPORT['table1'][sp][l][0] for sp in SPLITS for l in ('CREATE','DESTROY','MOVE'))}/9", t1_imp_ok)
check("R05", "Tổng cặp vàng D×C (3)", 533, len(PAIRS), len(PAIRS) == 533)
check("R06", "Cặp vàng train/dev/test", "378/58/97", "/".join(str(PAIRS_SPLIT[sp]) for sp in SPLITS),
      all(PAIRS_SPLIT[sp] == REPORT["gold_pairs_split"][sp] for sp in SPLITS))
check("R07", "Điểm quyết định DESTROY/CREATE (tr.2)", "57/76", f"{DP_STATS['DESTROY']['n']}/{DP_STATS['CREATE']['n']}",
      (DP_STATS["DESTROY"]["n"], DP_STATS["CREATE"]["n"]) == (57, 76))
ac = (DP_STATS["DESTROY"]["avg_candidates"] or 0, DP_STATS["CREATE"]["avg_candidates"] or 0)
check("R08", "Số ứng viên TB DESTROY/CREATE (tr.3, làm tròn)", "2.6/2.9", f"{ac[0]:.2f}/{ac[1]:.2f}",
      abs(ac[0] - 2.6) <= 0.06 and abs(ac[1] - 2.9) <= 0.06, "WARN", "báo cáo có vẻ cắt bớt chữ số thay vì làm tròn")
ra = (100 * (DP_STATS["DESTROY"]["random_acc"] or 0), 100 * (DP_STATS["CREATE"]["random_acc"] or 0))
check("R09", "Chọn ngẫu nhiên kỳ vọng DESTROY/CREATE (Bảng 3)", "51%/40%", f"{ra[0]:.1f}%/{ra[1]:.1f}%",
      round(ra[0]) == 51 and round(ra[1]) == 40)
lift_ok = all(abs(LIFT[k]["lift"] - v[0]) < 0.051 and LIFT[k]["n_implicit_A_with_B"] == v[1] for k, v in REPORT["lift"].items())
check("R10", "Bảng 2: lift + số ô (9 ô)", "khớp 9/9",
      f"khớp {sum(abs(LIFT[k]['lift'] - v[0]) < 0.051 and LIFT[k]['n_implicit_A_with_B'] == v[1] for k, v in REPORT['lift'].items())}/9", lift_ok)
check("R11", "PP1: câu hỏi nhãn 4ΣTn / cặp ΣTn(n−1) (6)(7)", "52936/48070", f"{PP2['label_questions']}/{PP2['all_pairs']}",
      (PP2["label_questions"], PP2["all_pairs"]) == (52936, 48070))
check("R12", "PP2 với tồn tại VÀNG: số cặp còn lại / cặp vàng giữ lại (12)", "11144/533", f"{PP2['kept_gold_existence']}/{PP2['gold_kept']}",
      (PP2["kept_gold_existence"], PP2["gold_kept"]) == (11144, 533))
check("R13", "Lọc đồng xuất hiện: % cặp vàng / % cặp nguồn ngầm (tr.4)", "74.7/18.9",
      f"{COOC_PCT['any_sentence_gold']:.1f}/{COOC_PCT['any_sentence_src_implicit']:.1f}",
      round(COOC_PCT["any_sentence_gold"], 1) == 74.7 and round(COOC_PCT["any_sentence_src_implicit"], 1) == 18.9, "WARN",
      "chỉ khớp nếu hiểu 'đồng xuất hiện' = cùng được nhắc trong MỘT câu bất kỳ của đoạn; nếu hiểu là cùng câu t thì "
      f"{COOC_PCT['same_sentence_gold']:.1f}/{COOC_PCT['same_sentence_src_implicit']:.1f}")
check("R14", "Số thực thể TB mỗi đoạn (tr.6)", 3.9, PP4["mean_entities"], round(PP4["mean_entities"], 1) == 3.9)

# --- D: phát hiện cần ghi nhận (không chặn)
check("D01", "Số conversion vàng theo bộ chấm CHÍNH THỨC ≠ số cặp D×C", "khác 533 (định nghĩa khác)",
      f"{len(OFFICIAL_CONV)} ({'/'.join(str(OFFICIAL_SPLIT[sp]) for sp in SPLITS)})", False, "WARN",
      "evaluator gộp mọi D,C của một câu thành 1 conversion + luật ghép câu kế tiếp + chấm Jaccard mềm, macro theo đoạn. "
      "Bản sao này đếm cả thực thể trùng tên; evaluator gốc gộp chúng ở 4 đoạn train → notebook 01 đối chiếu với mã gốc")
check("D02", "Cặp ở câu 1×1 (cặp được quyết định hoàn toàn bởi nhãn)", "ghi nhận",
      f"{PAIR_STATS['sentences_1x1']}/{PAIR_STATS['sentences_with_pairs']} câu; {len(PAIRS) - PAIR_STATS['pairs_from_multi_entity_sentences']}/{len(PAIRS)} cặp",
      True, "INFO", "với định nghĩa D×C, tập cặp là hàm của nhãn CREATE/DESTROY")
check("D03", "Không có đoạn n>8 (giải mã chính xác 4^n khả thi)", 0, PP4["paragraphs_n_gt_8"], PP4["paragraphs_n_gt_8"] == 0, "WARN")
check("D04", "Độ nhạy cờ nhắc (extended −es/−ies) — số ô ngầm thay đổi", "ghi nhận", sum(SENS.values()), True, "INFO", str(SENS_D))

n_fail = sum(c["status"] == "FAIL" for c in CHECKS)
n_warn = sum(c["status"] == "WARN" for c in CHECKS)
GATE_STATUS = "SMOKE_OK" if (SMOKE_TEST and n_fail == 0) else ("PASS" if n_fail == 0 else "FAIL")
w = max(len(c["check"]) for c in CHECKS)
print(f"{'ID':4s} {'TRẠNG THÁI':11s} {'KIỂM TRA':{w}s}  KỲ VỌNG → QUAN SÁT")
for c in CHECKS:
    print(f"{c['id']:4s} {c['status']:11s} {c['check']:{w}s}  {c['expected']} → {c['observed']}")
print(f"\nFAIL={n_fail}  WARN={n_warn}  →  GATE_STATUS = {GATE_STATUS}")

ID   TRẠNG THÁI  KIỂM TRA                                                              KỲ VỌNG → QUAN SÁT
S01  PASS        Kiểm thử đơn vị                                                       16/16 → 16/16
S02  PASS        SHA-256 file gốc khớp giá trị ghim                                    8/8 → 8/8
S03  PASS        Không có lỗi cấu trúc (độ dài trạng thái, câu rỗng, trạng thái rỗng)  0 → 0
S04  WARN        Thực thể trùng tên trong cùng đoạn                                    0 → 6
S05  PASS        Mã đoạn duy nhất, không chéo tập, khớp file phân chia                 0/0/0 → 0/0/0
S06  PASS        Không có chủ đề (prompt) nằm ở >1 tập                                 0 → 0
S07  PASS        Không có đoạn văn trùng nguyên văn giữa các tập                       0 → 0
S08  PASS        Câu trùng nguyên văn giữa các tập (thông tin)                         ghi nhận → 18
S09  PASS        Nhãn vàng không vi phạm ràng buộc tồn tại                             0 → 0
S10  PASS        Hành động su

## 15. Lưu đầu ra (dữ liệu chuẩn, báo cáo, manifest, zip)

In [15]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_csv(path, rows, fields=None):
    fields = fields or (list(rows[0].keys()) if rows else [])
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v) for k, v in r.items()})

P = OUT / "data/processed"
for sp in SPLITS:
    write_jsonl(P / f"{sp}.jsonl", BY_SPLIT[sp])
    # Đầu vào cho mô hình: KHÔNG có trạng thái/nhãn → tránh lộ đáp án ở các notebook Qwen/mô hình
    write_jsonl(P / f"model_inputs_{sp}.jsonl",
                [{"paragraph_id": r["paragraph_id"], "split": sp, "sentences": r["sentences"], "entities": r["entities"]} for r in BY_SPLIT[sp]])
write_jsonl(P / "gold_pairs_report.jsonl", PAIRS)
write_jsonl(P / "official_q3_gold_replica.jsonl", OFFICIAL_CONV)
write_jsonl(P / "decision_points.jsonl", DPS)
cells = []
for r in RECS:
    for i, ent in enumerate(r["entities"]):
        for t in range(1, len(r["sentences"]) + 1):
            cells.append({"paragraph_id": r["paragraph_id"], "split": r["split"], "sentence_id": t, "entity_idx": i,
                          "entity": ent, "state_before": r["states"][i][t - 1], "state_after": r["states"][i][t],
                          "exists_before": int(r["existence"][i][t - 1]), "exists_after": int(r["existence"][i][t]),
                          "label": r["labels"][i][t - 1], "mentioned": int(r["mentions"][i][t - 1]),
                          "implicit": int(r["implicit"][i][t - 1])})
write_csv(P / "cells.csv", cells)
assert not any(k in json.loads(open(P / "model_inputs_dev.jsonl", encoding="utf-8").readline())
               for k in ("states", "labels", "existence", "implicit")), "model_inputs bị lộ nhãn!"

write_csv(OUT / "data/splits/official_split.csv",
          [{"paragraph_id": r["paragraph_id"], "split": r["split"], "topic": r["topic"]} for r in RECS])
write_csv(OUT / "data/splits/cv5_by_topic.csv",
          [{"paragraph_id": r["paragraph_id"], "official_split": r["split"], "topic": r["topic"], "fold": FOLDS[r["paragraph_id"]]} for r in RECS])
AUDIT_FIELDS = ["audit_group", "split", "paragraph_id", "sentence_id", "source", "target", "source_implicit", "target_implicit",
                "n_destroy_in_sentence", "n_create_in_sentence", "sentence_text", "paragraph_text",
                "annotator_A_label", "annotator_A_note", "annotator_B_label", "annotator_B_note", "resolved_label"]
rec_by_pid = {r["paragraph_id"]: r for r in RECS}
for a in AUDIT:
    rr = rec_by_pid[a["paragraph_id"]]
    a["sentence_text"] = rr["sentences"][a["sentence_id"] - 1]
    a["paragraph_text"] = " ".join(f"[{k}] {s}" for k, s in enumerate(rr["sentences"], 1))
    for k in AUDIT_FIELDS[-5:]:
        a[k] = ""
write_csv(OUT / "data/audit/gold_audit_sample.csv", AUDIT, AUDIT_FIELDS)

write_csv(OUT / "reports/gate_checks.csv", CHECKS)
SUMMARY = {
    "run_tag": RUN_TAG, "core_version": pc.CORE_VERSION, "gate_status": GATE_STATUS, "n_fail": n_fail, "n_warn": n_warn,
    "smoke_test": SMOKE_TEST, "seed": SEED, "env": ENV, "raw_sources": RAW_SOURCE, "checksums": CHECKSUM,
    "counts": COUNTS, "table1_implicit_over_total": {sp: {l: list(v) for l, v in TABLE1[sp].items()} for sp in SPLITS},
    "pair_stats": PAIR_STATS, "decision_points": DP_STATS, "pp1_pp2": dict(PP2), "cooccurrence_pct": COOC_PCT,
    "lift": {f"{a}|{b}": v for (a, b), v in LIFT.items()}, "pp4_feasibility": PP4, "mention_sensitivity": SENS_D,
    "duplicate_entity_paragraphs": sorted({p for p, _ in DUP_ISSUES}), "cross_split_duplicate_sentences": SENT_CROSS,
    "folds": {str(f): dict(v) for f, v in sorted(FOLD_STATS.items())}, "checks": CHECKS,
    "runtime_sec": round(time.time() - T0, 1), "figures": FIGS,
}
(OUT / "reports/audit_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False), encoding="utf-8")

md_lines = [f"# dataset_validation — {RUN_TAG}", "",
            f"- GATE_STATUS: **{GATE_STATUS}** (FAIL={n_fail}, WARN={n_warn}); SMOKE_TEST={SMOKE_TEST}",
            f"- Nguồn: allenai/propara@{PIN_PROPARA[:7]} (grids.v1), allenai/aristo-leaderboard@{PIN_LEADER[:7]} (answers.tsv)",
            f"- Môi trường: Python {ENV['python']}, {ENV['platform']}, on_kaggle={ENV['on_kaggle']}", "",
            "| ID | Trạng thái | Kiểm tra | Kỳ vọng | Quan sát | Ghi chú |", "|---|---|---|---|---|---|"]
md_lines += [f"| {c['id']} | {c['status']} | {c['check']} | {c['expected']} | {c['observed']} | {c['note']} |" for c in CHECKS]
(OUT / "reports/dataset_validation.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")

manifest = {"run_tag": RUN_TAG, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "files": {}}
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.name != "manifest.json":
        manifest["files"][p.relative_to(OUT).as_posix()] = {"bytes": p.stat().st_size, "sha256": pc.sha256_file(str(p))}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")

ZIP = WORK_ROOT / f"{RUN_TAG}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, Path(RUN_TAG) / p.relative_to(OUT))
print(f"Đã ghi {len(manifest['files'])} file vào {OUT}\nZip: {ZIP} ({ZIP.stat().st_size/1e6:.2f} MB)")

Đã ghi 20 file vào C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\runs\r1792352508\propara_audit_v1
Zip: C:\Users\ADMIN\.claude\jobs\3967a9d0\tmp\runs\r1792352508\propara_audit_v1.zip (0.38 MB)


## 16. Kết quả để gửi lại (sao chép toàn bộ khối dưới đây)

In [16]:
print("=== GỬI LẠI ===")
print(f"notebook=00_propara_data_audit run_tag={RUN_TAG} core={pc.CORE_VERSION} GATE_STATUS={GATE_STATUS} FAIL={n_fail} WARN={n_warn} smoke={SMOKE_TEST}")
print(f"env: python={ENV['python']} on_kaggle={ENV['on_kaggle']} gpu={ENV['gpu_present']} internet={ENV['internet']} runtime={SUMMARY['runtime_sec']}s")
print("nguồn:", sorted({v.split(':')[0] for v in RAW_SOURCE.values()}), "| checksum khớp:", sum(v['match'] for v in CHECKSUM.values()), "/8")
for c in CHECKS:
    if c["status"] != "PASS":
        print(f"  {c['id']} {c['status']}: {c['check']} | {c['expected']} → {c['observed']}")
print("cặp:", PAIR_STATS["pairs_by_split"], "| official conv (bản sao):", PAIR_STATS["official_q3_conversions_replica_by_split"])
print("điểm QĐ:", {k: (v['n'], v['by_split']) for k, v in DP_STATS.items()})
print("=== HẾT ===")
if GATE_STATUS == "FAIL":
    raise RuntimeError("Cổng nghiệm thu FAIL. Xem bảng ở mục 14; output đã được lưu để chẩn đoán.")

=== GỬI LẠI ===
notebook=00_propara_data_audit run_tag=propara_audit_v1 core=core_v1 GATE_STATUS=PASS FAIL=0 WARN=2 smoke=False
env: python=3.14.4 on_kaggle=False gpu=True internet=True runtime=1.4s
nguồn: ['input'] | checksum khớp: 8 /8
  S04 WARN: Thực thể trùng tên trong cùng đoạn | 0 → 6
  D01 WARN: Số conversion vàng theo bộ chấm CHÍNH THỨC ≠ số cặp D×C | khác 533 (định nghĩa khác) → 485 (354/49/82)
cặp: {'train': 378, 'dev': 58, 'test': 97} | official conv (bản sao): {'train': 354, 'dev': 49, 'test': 82}
điểm QĐ: {'DESTROY': (57, {'train': 37, 'dev': 10, 'test': 10}), 'CREATE': (76, {'train': 56, 'dev': 8, 'test': 12})}
=== HẾT ===
